In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/dp'
%cd {PROJECT_PATH}/code

In [ ]:
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import os

# Qwen/Qwen2.5-0.5B-Instruct | Qwen/Qwen2.5-1.5B-Instruct | Qwen/Qwen2.5-7B-Instruct
os.environ['HF_MODEL_NAME'] = 'Qwen/Qwen2.5-0.5B-Instruct'
os.environ['HF_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['LLM_TYPE'] = 'hf'
os.environ['LLM_MAX_NEW_TOKENS'] = '24'

os.environ['K'] = '3'
os.environ['T_MAX'] = '4'
os.environ['N'] = '24'

os.environ['TASK_TYPE'] = 'word_problem'
os.environ['TASK_NUM_OPERANDS'] = '6'
os.environ['TASK_MAX_VALUE'] = '20'
os.environ['TASK_OPS'] = '+,-,*'

os.environ['RANDOM_SEED'] = '0'

In [ ]:
!python scripts/run_rollout.py --n 30 --out data/rollouts.jsonl

In [ ]:
import json
from pathlib import Path

from shapley_dreamer.training.metrics import format_summary, summarize

rollouts = [json.loads(line) for line in Path('data/rollouts.jsonl').open()]
print(format_summary(summarize(rollouts)))
print()
for ep in rollouts[-5:]:
    print(f"--- ep {ep['episode']} | gt={ep['gt']!r} | r={ep['success']} ---")
    for t, (cells, action) in enumerate(zip(ep['cells_history'][1:], ep['actions'])):
        print(f'  step {t}: action={action} cells={cells}')
    print()

In [ ]:
import json
from pathlib import Path

from shapley_dreamer import settings
from shapley_dreamer.training.dreamer_loop import run_comparison

results = run_comparison(
    settings_module=settings,
    n_collect=500,
    wm_epochs=30,
    ac_updates=300,
    eval_every=30,
    eval_episodes=30,
)
Path('data').mkdir(exist_ok=True)
with open('data/curves.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
import json
import matplotlib.pyplot as plt

with open('data/curves.json') as f:
    results = json.load(f)

fig, ax = plt.subplots(figsize=(8, 5))
for mode, curve in results['curves'].items():
    steps = [p['step'] for p in curve]
    rates = [p['success_rate'] for p in curve]
    ax.plot(steps, rates, marker='o', label=mode)

ax.axhline(results['settings']['random_baseline_success'], linestyle='--', color='gray', label='random')
ax.set_xlabel('AC update step')
ax.set_ylabel('success_rate (real env)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
!python -m pytest ../tests/ -q